# THINGS Odd-One-Out: Human–Model Alignment — Colab Runner

Runs the **full pipeline** on a Colab GPU — everything is scripted, including the
CC0 images (no login or agreement). The smoke-test section validates Phases 3–5
**without images** using synthetic features; section 4 is the real run.

**Tip:** Runtime → Change runtime type → GPU (T4).

## 1. Clone + install

In [ ]:
!git clone https://github.com/Sudarssan-N/VisionModal-HumalObjectSimilarity.git
%cd VisionModal-HumalObjectSimilarity
!pip install -q -r requirements.txt

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Download + prepare behavioral data (scripted, ~412 MB)

In [ ]:
!python src/download_data.py
!cd data/raw && unzip -q -o osfstorage-archive.zip -d extracted
!cd data/raw/extracted && unzip -q -o full_triplet_dataset.zip -d triplets
!python src/prepare_data.py

## 3. Smoke test — validate Phases 3–5 WITHOUT images

Fabricates features that are a noisy linear function of the real human
embedding, then runs the real training/transfer/analysis scripts. Confirms the
pipeline works on this machine before investing in feature extraction.

In [ ]:
!python src/smoke_test.py --force

## 4. Real run: images → features → results

The CC0 reference images (one per concept, 1.1 GB) download automatically — no
login or agreement needed.

In [ ]:
# Download + organize the 1,854 CC0 images (1.1 GB)
!python src/download_data.py --images
!python src/organize_images.py data/raw/images_cc0
!python src/data.py

In [ ]:
# Phase 1 + 2: extract embeddings (frozen, GPU), zero-shot baseline
!python src/extract_features.py
!python src/zeroshot_eval.py

In [ ]:
# Phase 3-5: alignment, transfer, analysis
!python src/train_transform.py
!python src/transfer.py
!python src/analysis.py

In [ ]:
# Assemble all results into a single markdown report
!python src/make_report.py

## 5. Robustness (seeds / λ sweep / image-disjoint split)

Confirms the aligned numbers are stable and not a leakage artifact. Adds a
robustness section to `results/REPORT.md`.

In [ ]:
!python src/run_robustness.py
!python src/make_report.py

## 6. Compile the paper (PDF)

Builds the NeurIPS-format preprint with the real figures. Download `neurips_2024.sty`
into `paper/` for the official format (optional — falls back to a clean article style).

In [ ]:
!apt-get -qq install texlive-latex-extra texlive-bibtex-extra texlive-fonts-recommended >/dev/null
import os; os.makedirs('paper/figures', exist_ok=True)
!cp results/transfer_heatmap.png results/rsa.png paper/figures/ 2>/dev/null || true
%cd paper
!pdflatex -interaction=nonstopmode main >/dev/null && bibtex main >/dev/null && pdflatex -interaction=nonstopmode main >/dev/null && pdflatex -interaction=nonstopmode main >/dev/null
%cd ..
from google.colab import files
files.download('paper/main.pdf')